# EAMSNet (MobileNetV2)

Arsitektur EAMSNet dengan backbone **MobileNetV2** dan tiga modul yang diusulkan: **ATDAM**, **MSDA**, dan **EABRM**. Modul Deformable Attention (DAT) tidak digunakan pada versi ini.

Checkpoint disimpan ke `best_eamsnet.pth`.

## 1. Environment Setup

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False
    print('timm tidak ditemukan -> akan memakai ResNet-50 torchvision sebagai fallback.')
    print('Untuk ConvNeXt (sedikit lebih akurat): pip install timm')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. LEVIR-CD Dataset (dengan augmentasi sedikit lebih kaya)

In [ ]:
class LEVIRCDDataset(Dataset):
    def __init__(self, root_dir, split='train', img_size=256, augment=True):
        self.root_dir=os.path.join(root_dir,split)
        self.augment=augment and (split=='train')
        self.img_size=img_size
        self.img_A_dir=os.path.join(self.root_dir,'A')
        self.img_B_dir=os.path.join(self.root_dir,'B')
        self.label_dir=os.path.join(self.root_dir,'label')
        self.filenames=sorted([f for f in os.listdir(self.img_A_dir) if f.endswith(('.png','.jpg','.tif'))])
        self.norm=transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        print(f'[{split.upper()}] {len(self.filenames)} pairs loaded')

    def __len__(self): return len(self.filenames)

    def _augment(self, imgA, imgB, label):
        if np.random.random() > 0.5:
            scale = np.random.uniform(1.0, 1.3)
            ns = int(self.img_size*scale)
            imgA=transforms.functional.resize(imgA,(ns,ns))
            imgB=transforms.functional.resize(imgB,(ns,ns))
            label=transforms.functional.resize(label,(ns,ns),interpolation=transforms.InterpolationMode.NEAREST)
            top=np.random.randint(0,ns-self.img_size+1); left=np.random.randint(0,ns-self.img_size+1)
            imgA=transforms.functional.crop(imgA,top,left,self.img_size,self.img_size)
            imgB=transforms.functional.crop(imgB,top,left,self.img_size,self.img_size)
            label=transforms.functional.crop(label,top,left,self.img_size,self.img_size)
        if np.random.random()>0.5:
            imgA,imgB,label=(transforms.functional.hflip(x) for x in (imgA,imgB,label))
        if np.random.random()>0.5:
            imgA,imgB,label=(transforms.functional.vflip(x) for x in (imgA,imgB,label))
        angle=int(np.random.choice([0,90,180,270]))
        if angle>0:
            imgA=transforms.functional.rotate(imgA,angle)
            imgB=transforms.functional.rotate(imgB,angle)
            label=transforms.functional.rotate(label,angle)
        if np.random.random()>0.5:
            jitter=transforms.ColorJitter(0.3,0.3,0.2,0.05)
            imgA,imgB=jitter(imgA),jitter(imgB)
        if np.random.random()>0.7:
            blur=transforms.GaussianBlur(3,sigma=(0.1,1.2))
            imgA,imgB=blur(imgA),blur(imgB)
        if np.random.random()>0.5:
            imgA,imgB=imgB,imgA
        return imgA,imgB,label

    def __getitem__(self, idx):
        f=self.filenames[idx]
        imgA=Image.open(os.path.join(self.img_A_dir,f)).convert('RGB')
        imgB=Image.open(os.path.join(self.img_B_dir,f)).convert('RGB')
        label=Image.open(os.path.join(self.label_dir,f)).convert('L')
        imgA=transforms.functional.resize(imgA,(self.img_size,self.img_size))
        imgB=transforms.functional.resize(imgB,(self.img_size,self.img_size))
        label=transforms.functional.resize(label,(self.img_size,self.img_size),interpolation=transforms.InterpolationMode.NEAREST)
        if self.augment:
            imgA,imgB,label=self._augment(imgA,imgB,label)
        tA=self.norm(transforms.functional.to_tensor(imgA))
        tB=self.norm(transforms.functional.to_tensor(imgB))
        tL=(transforms.functional.to_tensor(label)>0.5).float()
        return tA,tB,tL

In [ ]:
BACKBONE = 'mobilenet_v2'
WIDTH    = 'lite'

DATA_ROOT   = 'LEVIR-CD-256'
IMG_SIZE    = 256
BATCH_SIZE  = 16 if IMG_SIZE==256 else 4

NUM_WORKERS = 0

train_dataset = LEVIRCDDataset(DATA_ROOT,'train',IMG_SIZE,augment=True)
val_dataset   = LEVIRCDDataset(DATA_ROOT,'val',  IMG_SIZE,augment=False)
test_dataset  = LEVIRCDDataset(DATA_ROOT,'test', IMG_SIZE,augment=False)

_pw = NUM_WORKERS > 0
_pm = torch.cuda.is_available()
train_loader = DataLoader(train_dataset,BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,
                          pin_memory=_pm,drop_last=True,persistent_workers=_pw)
val_loader   = DataLoader(val_dataset,BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
                          pin_memory=_pm,persistent_workers=_pw)
test_loader  = DataLoader(test_dataset,BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
                          pin_memory=_pm,persistent_workers=_pw)
print(f'IMG_SIZE={IMG_SIZE} BATCH={BATCH_SIZE} WORKERS={NUM_WORKERS} | '
      f'Train:{len(train_loader)} Val:{len(val_loader)} Test:{len(test_loader)}')

## 3. Siamese Encoder yang Diperkuat (ConvNeXt-Tiny / ResNet-50, Output Stride 16)

Encoder ini mengeluarkan 4 skala fitur dengan channel yang dinormalisasi ke **[64, 128, 256, 512]**
melalui proyeksi 1x1, sehingga seluruh modul novel (ATDAM/EABRM/MSDA) dan decoder dari versi asli
**tidak perlu diubah channel-nya**. Maxpool/downsampling stem dihapus -> output stride 16, bukan 32,
memberi feature map 2x lebih detail pada bangunan kecil.


In [ ]:
class SiameseEncoder(nn.Module):
    OUT_CH = [64, 128, 256, 512]

    def __init__(self, backbone='convnext_tiny', pretrained=True, out_ch=None):
        super().__init__()
        self.backbone_name = backbone
        self.OUT_CH = list(out_ch) if out_ch is not None else [64, 128, 256, 512]

        if backbone == 'convnext_tiny':
            import timm
            self.body = timm.create_model('convnext_tiny', pretrained=pretrained,
                                          features_only=True, out_indices=(0,1,2,3),
                                          drop_path_rate=0.2)
            raw_ch = self.body.feature_info.channels()
        elif backbone == 'mobilen_v2' or backbone == 'mobilenet_v2':
            mnet = models.mobilenet_v2(
                weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
            feats = mnet.features
            self.m_stage1 = feats[0:4]
            self.m_stage2 = feats[4:7]
            self.m_stage3 = feats[7:14]
            self.m_stage4 = feats[14:]
            raw_ch = [24, 32, 96, 1280]
        elif backbone == 'resnet50':
            resnet = models.resnet50(
                weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
            resnet.maxpool = nn.Identity()
            self.stem   = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
            self.layer1 = resnet.layer1
            self.layer2 = resnet.layer2
            self.layer3 = resnet.layer3
            self.layer4 = resnet.layer4
            raw_ch = [256, 512, 1024, 2048]
        else:
            raise ValueError(f'Backbone tidak dikenal: {backbone}')

        self.proj = nn.ModuleList([
            nn.Sequential(nn.Conv2d(raw_ch[i], self.OUT_CH[i], 1, bias=False),
                          nn.BatchNorm2d(self.OUT_CH[i]), nn.ReLU(True))
            for i in range(4)])

    def forward(self, x):
        if self.backbone_name == 'convnext_tiny':
            feats = self.body(x)
        elif self.backbone_name in ('mobilen_v2', 'mobilenet_v2'):
            f1 = self.m_stage1(x)
            f2 = self.m_stage2(f1)
            f3 = self.m_stage3(f2)
            f4 = self.m_stage4(f3)
            feats = [f1, f2, f3, f4]
        else:
            x  = self.stem(x)
            f1 = self.layer1(x)
            f2 = self.layer2(f1)
            f3 = self.layer3(f2)
            f4 = self.layer4(f3)
            feats = [f1, f2, f3, f4]
        return [self.proj[i](feats[i]) for i in range(4)]

## 5. Modul Novel Asli (ATDAM, EABRM, MSDA) — dipertahankan, channel sudah cocok

In [ ]:
class ATDAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        r = max(channels // reduction, 8)
        self.ch_pool = nn.AdaptiveAvgPool2d(1)
        self.ch_fc   = nn.Sequential(
            nn.Linear(channels * 3, r), nn.ReLU(inplace=True),
            nn.Linear(r, channels),     nn.Sigmoid())
        self.sp_conv = nn.Sequential(
            nn.Conv2d(6, 16, 7, padding=3, bias=False), nn.BatchNorm2d(16), nn.ReLU(True),
            nn.Conv2d(16, 1, 7, padding=3, bias=False), nn.Sigmoid())
        self.gate = nn.Sequential(
            nn.Conv2d(channels, channels, 1, bias=False),
            nn.BatchNorm2d(channels), nn.Sigmoid())
        self.proj = nn.Sequential(
            nn.Conv2d(channels, channels, 1, bias=False),
            nn.BatchNorm2d(channels))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, f1, f2):
        B, C = f1.shape[:2]
        diff = f1 - f2
        abs_diff = torch.abs(diff)
        ch_in = torch.cat([self.ch_pool(f1).view(B, C),
                           self.ch_pool(f2).view(B, C),
                           self.ch_pool(abs_diff).view(B, C)], dim=1)
        ch_att = self.ch_fc(ch_in).view(B, C, 1, 1)
        def _sp(t): return torch.cat([t.mean(1, True), t.max(1, True)[0]], 1)
        sp_att = self.sp_conv(torch.cat([_sp(f1), _sp(f2), _sp(abs_diff)], 1))
        attended = diff * ch_att * sp_att * self.gate(abs_diff)
        return self.relu(self.proj(attended) + diff)

class EABRM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.edge_x = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.edge_y = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self._init_sobel()
        mid = max(channels // 4, 16)
        self.enhance = nn.Sequential(
            nn.Conv2d(channels*2, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(True),
            nn.Conv2d(mid, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(True))
        self.gate = nn.Sequential(
            nn.Conv2d(channels*2, channels, 1, bias=False), nn.BatchNorm2d(channels), nn.Sigmoid())
        self.out = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(True))

    def _init_sobel(self):
        sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32)
        sy = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32)
        with torch.no_grad():
            self.edge_x.weight.copy_(sx.view(1,1,3,3).repeat(self.edge_x.weight.shape[0],1,1,1))
            self.edge_y.weight.copy_(sy.view(1,1,3,3).repeat(self.edge_y.weight.shape[0],1,1,1))

    def _edge(self, feat):
        return self.enhance(torch.cat([self.edge_x(feat), self.edge_y(feat)], 1))

    def forward(self, semantic, f1, f2):
        edge_diff = torch.abs(self._edge(f1) - self._edge(f2))
        g = self.gate(torch.cat([semantic, edge_diff], 1))
        return self.out(semantic + g * edge_diff)

class MSDA(nn.Module):
    def __init__(self, channels):
        super().__init__()
        m = channels // 4
        self.b1 = nn.Sequential(nn.Conv2d(channels, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(True))
        self.b2 = nn.Sequential(nn.Conv2d(channels, m, 3, padding=2, dilation=2, bias=False), nn.BatchNorm2d(m), nn.ReLU(True))
        self.b3 = nn.Sequential(nn.Conv2d(channels, m, 3, padding=4, dilation=4, bias=False), nn.BatchNorm2d(m), nn.ReLU(True))
        self.b4 = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(True))
        self.fuse = nn.Sequential(nn.Conv2d(m*4, channels, 1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(True), nn.Dropout2d(0.1))

    def forward(self, x):
        H, W = x.shape[2:]
        g = F.interpolate(self.b4(x), (H, W), mode='bilinear', align_corners=True)
        return self.fuse(torch.cat([self.b1(x), self.b2(x), self.b3(x), g], 1)) + x

## 6. Decoder Block (dari EAMSNet asli)

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up     = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.reduce = nn.Conv2d(in_ch, in_ch // 2, 1, bias=False)
        self.conv   = nn.Sequential(
            nn.Conv2d(in_ch//2 + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(True))

    def forward(self, x, skip):
        x = self.reduce(self.up(x))
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, skip.shape[2:], mode='bilinear', align_corners=True)
        return self.conv(torch.cat([x, skip], 1))

## 7. EAMSNet — Arsitektur Lengkap

Alur:
```
T1, T2 -> Siamese Encoder (MobileNetV2, channel dinormalisasi ke [64,128,256,512])
       -> ATDAM di tiap skala (atensi selisih temporal)
       -> MSDA di bottleneck
       -> U-Net Decoder + EABRM di tiap level
       -> Change Map (+ deep supervision saat training)
```

In [ ]:
class EAMSNetPP(nn.Module):
    def __init__(self, backbone='mobilenet_v2', pretrained=True, width='full'):
        super().__init__()
        if isinstance(width, (list, tuple)):
            ch = list(width)
        elif width == 'lite':
            ch = [32, 64, 128, 256]
        else:
            ch = [64, 128, 256, 512]
        self.ch = ch

        self.encoder = SiameseEncoder(backbone, pretrained, out_ch=ch)
        self.atdam   = nn.ModuleList([ATDAM(c) for c in ch])

        self.msda   = MSDA(ch[3])
        self.dec3   = DecoderBlock(ch[3], ch[2], ch[2])
        self.dec2   = DecoderBlock(ch[2], ch[1], ch[1])
        self.dec1   = DecoderBlock(ch[1], ch[0], ch[0])
        self.eabrm3 = EABRM(ch[2])
        self.eabrm2 = EABRM(ch[1])
        self.eabrm1 = EABRM(ch[0])

        self.head = nn.Sequential(
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True),
            nn.Conv2d(ch[0], 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Dropout2d(0.1), nn.Conv2d(32, 1, 1))

        self.ds3 = nn.Conv2d(ch[2], 1, 1)
        self.ds2 = nn.Conv2d(ch[1], 1, 1)
        self.ds1 = nn.Conv2d(ch[0], 1, 1)

    def forward(self, imgA, imgB):
        H, W = imgA.shape[2:]
        fA = self.encoder(imgA)
        fB = self.encoder(imgB)

        diffs = [self.atdam[i](fA[i], fB[i]) for i in range(4)]
        diffs[3] = self.msda(diffs[3])

        def _match(skipA, skipB, ref):
            if skipA.shape[2:] != ref.shape[2:]:
                skipA = F.interpolate(skipA, ref.shape[2:], mode='bilinear', align_corners=True)
                skipB = F.interpolate(skipB, ref.shape[2:], mode='bilinear', align_corners=True)
            return skipA, skipB

        s3 = self.dec3(diffs[3], diffs[2]); a2,b2 = _match(fA[2],fB[2],s3); d3 = self.eabrm3(s3, a2, b2)
        s2 = self.dec2(d3, diffs[1]);       a1,b1 = _match(fA[1],fB[1],s2); d2 = self.eabrm2(s2, a1, b1)
        s1 = self.dec1(d2, diffs[0]);       a0,b0 = _match(fA[0],fB[0],s1); d1 = self.eabrm1(s1, a0, b0)

        out = self.head(d1)
        if out.shape[2:] != (H, W):
            out = F.interpolate(out, (H, W), mode='bilinear', align_corners=True)

        aux = []
        if self.training:
            up = lambda t: F.interpolate(t, (H,W), mode='bilinear', align_corners=True)
            aux = [up(self.ds3(d3)), up(self.ds2(d2)), up(self.ds1(d1))]
        return out, aux

model = EAMSNetPP(BACKBONE, pretrained=True, width=WIDTH).to(device)
model.train()
with torch.no_grad():
    o, a = model(torch.randn(2,3,256,256).to(device), torch.randn(2,3,256,256).to(device))
print(f'Backbone={BACKBONE}, width={WIDTH}')
print(f'Output: {o.shape}  |  Aux: {[tuple(x.shape) for x in a]}')
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M')

## 8. Hybrid Loss (Focal + Dice + Edge-Aware + Lovász) dengan Deep Supervision

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2.0, smooth=0.02):
        super().__init__(); self.alpha=alpha; self.gamma=gamma; self.smooth=smooth
    def forward(self, pred, target):
        pred,target=pred.float(),target.float()
        target=target*(1-self.smooth)+0.5*self.smooth
        p=torch.sigmoid(pred).clamp(1e-6,1-1e-6)
        bce=-(target*torch.log(p)+(1-target)*torch.log(1-p))
        pt=target*p+(1-target)*(1-p)
        w=(target*self.alpha+(1-target)*(1-self.alpha))*(1-pt)**self.gamma
        return (w*bce).mean()

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1.0):
        super().__init__(); self.a=alpha; self.b=beta; self.smooth=smooth
    def forward(self, pred, target):
        p=torch.sigmoid(pred.float()); t=target.float()
        tp=(p*t).sum(dim=(2,3))
        fp=(p*(1-t)).sum(dim=(2,3))
        fn=((1-p)*t).sum(dim=(2,3))
        tv=(tp+self.smooth)/(tp+self.a*fp+self.b*fn+self.smooth)
        return 1-tv.mean()

class EdgeAwareLoss(nn.Module):
    def __init__(self, edge_weight=3.0):
        super().__init__(); self.ew=edge_weight
    def forward(self, pred, target):
        p=torch.sigmoid(pred.float()).clamp(1e-6,1-1e-6); target=target.float()
        edge=F.max_pool2d(target,3,1,1)-(1-F.max_pool2d(1-target,3,1,1))
        w=1.0+(self.ew-1.0)*edge
        bce=-(target*torch.log(p)+(1-target)*torch.log(1-p))
        return (w*bce).mean()

def lovasz_hinge(logits, labels):
    logits=logits.reshape(-1); labels=labels.reshape(-1).float()
    if labels.sum()==0: return logits.mean()*0.0
    signs=2.0*labels-1.0; errors=1.0-logits*signs
    errors_sorted,perm=torch.sort(errors,descending=True)
    gt_sorted=labels[perm]; p=len(gt_sorted); gts=gt_sorted.sum()
    inter=gts-gt_sorted.cumsum(0); union=gts+(1-gt_sorted).cumsum(0)
    jaccard=1.0-inter/union
    if p>1: jaccard[1:p]=jaccard[1:p]-jaccard[0:p-1]
    return torch.dot(F.relu(errors_sorted),jaccard)

class HybridLoss(nn.Module):
    def __init__(self, ds_weights=[0.3,0.2,0.1]):
        super().__init__()
        self.focal=FocalLoss(); self.tversky=TverskyLoss(); self.edge=EdgeAwareLoss()
        self.ds_w=ds_weights
    def _loss(self, pred, target):
        pred,target=pred.float(),target.float()
        return (self.focal(pred,target)+self.tversky(pred,target)
                +0.5*self.edge(pred,target)+0.5*lovasz_hinge(pred,target))
    def forward(self, main, aux_list, target):
        loss=self._loss(main,target)
        for i,aux in enumerate(aux_list):
            if i<len(self.ds_w): loss+=self.ds_w[i]*self._loss(aux,target)
        return loss

criterion=HybridLoss().to(device)

## 9. Metrics (Precision, Recall, F1, IoU, OA, Kappa)

In [ ]:
class CDMetrics:
    def __init__(self): self.reset()
    def reset(self): self.tp=self.fp=self.tn=self.fn=0
    def update_from_binary(self, pbin, target):
        p = pbin.long().cpu().numpy().flatten()
        t = target.long().cpu().numpy().flatten()
        self.tp += int(((p==1)&(t==1)).sum()); self.fp += int(((p==1)&(t==0)).sum())
        self.tn += int(((p==0)&(t==0)).sum()); self.fn += int(((p==0)&(t==1)).sum())
    def update(self, pred, target, thr=0.5):
        self.update_from_binary((torch.sigmoid(pred)>thr), target)
    def compute(self):
        e=1e-7
        P=self.tp/(self.tp+self.fp+e); R=self.tp/(self.tp+self.fn+e)
        F1=2*P*R/(P+R+e); IoU=self.tp/(self.tp+self.fp+self.fn+e)
        OA=(self.tp+self.tn)/(self.tp+self.fp+self.tn+self.fn+e)
        N=self.tp+self.fp+self.tn+self.fn+e
        pe=((self.tp+self.fp)*(self.tp+self.fn)+(self.tn+self.fp)*(self.tn+self.fn))/N**2
        K=(OA-pe)/(1-pe+e)
        return {k:v*100 for k,v in {'Precision':P,'Recall':R,'F1':F1,'IoU':IoU,'OA':OA,'Kappa':K}.items()}

## 10. Training Configuration (param-group: backbone / modul lain)

In [ ]:
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup, total, min_lr=1e-6):
        self.opt=optimizer; self.warmup=warmup; self.total=total; self.min_lr=min_lr
        self.base_lr=[g['lr'] for g in optimizer.param_groups]
    def step(self, epoch):
        if epoch<self.warmup: f=(epoch+1)/self.warmup
        else:
            prog=(epoch-self.warmup)/(self.total-self.warmup)
            f=self.min_lr/self.base_lr[0]+(1-self.min_lr/self.base_lr[0])*0.5*(1+np.cos(np.pi*prog))
        for g,lr0 in zip(self.opt.param_groups,self.base_lr): g['lr']=lr0*f

class ModelEMA:
    def __init__(self, model, decay_max=0.999, warmup_steps=2000):
        import copy
        self.ema=copy.deepcopy(model).eval()
        self.decay_max=decay_max; self.warmup_steps=warmup_steps; self.step_n=0
        for p in self.ema.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        self.step_n+=1
        d=min(self.decay_max, (1+self.step_n)/(10+self.step_n))
        for e,m in zip(self.ema.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(d).add_(m.detach(), alpha=1-d)
            else: e.copy_(m)

NUM_EPOCHS = 250
PATIENCE   = 60
WARMUP     = 5

decay, no_decay = [], []
enc_decay, enc_no = [], []
for n,p in model.named_parameters():
    if not p.requires_grad: continue
    is_enc = n.startswith('encoder.')
    if p.ndim<=1 or 'norm' in n.lower() or n.endswith('.bias'):
        (enc_no if is_enc else no_decay).append(p)
    else:
        (enc_decay if is_enc else decay).append(p)

WD = 1e-2
optimizer = optim.AdamW([
    {'params':enc_decay, 'lr':1e-4, 'weight_decay':WD},
    {'params':enc_no,    'lr':1e-4, 'weight_decay':0.0},
    {'params':decay,     'lr':3e-4, 'weight_decay':WD},
    {'params':no_decay,  'lr':3e-4, 'weight_decay':0.0},
])
scheduler=WarmupCosineScheduler(optimizer,WARMUP,NUM_EPOCHS)
scaler=GradScaler()
ema=ModelEMA(model, decay_max=0.999)

print(f'Epochs:{NUM_EPOCHS} Warmup:{WARMUP} Patience:{PATIENCE} | WD:{WD} | EMA:on')
print(f'Param groups: enc_decay={len(enc_decay)} enc_no={len(enc_no)} decay={len(decay)} no_decay={len(no_decay)}')

## 11. Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train(); total_loss=0; metrics=CDMetrics()
    for imgA,imgB,label in loader:
        imgA,imgB,label = imgA.to(device),imgB.to(device),label.to(device)
        optimizer.zero_grad()
        with autocast():
            out,aux = model(imgA,imgB)
            loss = criterion(out,aux,label)
        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad(); continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
        metrics.update(out.detach().float(),label)
    return total_loss/len(loader), metrics.compute()

def train_epoch_ema(model, ema, loader, criterion, optimizer, scaler, device):
    model.train(); total_loss=0; metrics=CDMetrics()
    for imgA,imgB,label in loader:
        imgA,imgB,label=imgA.to(device),imgB.to(device),label.to(device)
        optimizer.zero_grad()
        with autocast():
            out,aux=model(imgA,imgB)
            loss=criterion(out,aux,label)
        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad(); continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update()
        ema.update(model)
        total_loss+=loss.item()
        metrics.update(out.detach().float(),label)
    return total_loss/len(loader), metrics.compute()

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval(); total_loss=0; metrics=CDMetrics()
    for imgA,imgB,label in loader:
        imgA,imgB,label = imgA.to(device),imgB.to(device),label.to(device)
        with autocast():
            out,_ = model(imgA,imgB)
            loss = criterion._loss(out,label)
        total_loss += loss.item()
        metrics.update(out.float(),label)
    return total_loss/len(loader), metrics.compute()

@torch.no_grad()
def search_threshold(model, loader, device, grid=None):
    if grid is None: grid = np.arange(0.30, 0.71, 0.02)
    model.eval()
    best_thr, best_f1 = 0.5, -1
    stats = {round(t,3):[0,0,0] for t in grid}
    for imgA,imgB,label in loader:
        imgA,imgB = imgA.to(device),imgB.to(device)
        with autocast():
            out,_ = model(imgA,imgB)
        p = torch.sigmoid(out.float()).cpu().numpy().flatten()
        t = label.long().numpy().flatten()
        for thr in grid:
            pb = (p>thr)
            s = stats[round(thr,3)]
            s[0]+=int(((pb==1)&(t==1)).sum())
            s[1]+=int(((pb==1)&(t==0)).sum())
            s[2]+=int(((pb==0)&(t==1)).sum())
    for thr in grid:
        tp,fp,fn = stats[round(thr,3)]; e=1e-7
        P=tp/(tp+fp+e); R=tp/(tp+fn+e); F1=2*P*R/(P+R+e)
        if F1>best_f1: best_f1, best_thr = F1, float(thr)
    return best_thr, best_f1*100

def postprocess(prob_map, thr, min_area=4, fill_holes=True):
    try:
        from scipy import ndimage
    except ImportError:
        return (prob_map>thr).astype(np.uint8)
    binm=(prob_map>thr).astype(np.uint8)
    if fill_holes:
        binm=ndimage.binary_fill_holes(binm).astype(np.uint8)
    if min_area>0:
        lab,n=ndimage.label(binm)
        if n>0:
            sizes=ndimage.sum(np.ones_like(lab),lab,range(1,n+1))
            for k,sz in enumerate(sizes,1):
                if sz<min_area: binm[lab==k]=0
    return binm

@torch.no_grad()
def evaluate_full(model, loader, device, thr=0.5, tta=False, use_postproc=False):
    model.eval(); metrics=CDMetrics()
    for imgA,imgB,label in loader:
        imgA,imgB = imgA.to(device),imgB.to(device)
        with autocast():
            p = torch.sigmoid(model(imgA,imgB)[0].float())
            if tta:
                p1=torch.sigmoid(model(imgA.flip(3),imgB.flip(3))[0].float()).flip(3)
                p2=torch.sigmoid(model(imgA.flip(2),imgB.flip(2))[0].float()).flip(2)
                p3=torch.sigmoid(model(imgA.flip(2).flip(3),imgB.flip(2).flip(3))[0].float()).flip(2).flip(3)
                p=(p+p1+p2+p3)/4
        p_np = p.cpu().numpy()
        for b in range(p_np.shape[0]):
            pm = p_np[b,0]
            if use_postproc:
                pbin = postprocess(pm, thr)
                metrics.update_from_binary(torch.from_numpy(pbin), label[b])
            else:
                metrics.update_from_binary(torch.from_numpy((pm>thr).astype(np.uint8)), label[b])
    return metrics.compute()

## 12. Training Loop

In [ ]:
best_f1=0; patience_cnt=0; best_src='raw'
history={'tr_loss':[],'vl_loss':[],'tr_f1':[],'vl_f1':[],'vl_iou':[],'vl_p':[],'vl_r':[]}

print('='*70); print('  EAMSNet++ v2 Training on LEVIR-CD'); print('='*70)

for epoch in range(1,NUM_EPOCHS+1):
    t0=time.time(); scheduler.step(epoch-1)
    tr_loss,tr_m=train_epoch_ema(model,ema,train_loader,criterion,optimizer,scaler,device)

    vl_loss_raw, vl_raw = evaluate(model,      val_loader, criterion, device)
    vl_loss_ema, vl_ema = evaluate(ema.ema,    val_loader, criterion, device)
    if vl_ema['F1'] >= vl_raw['F1']:
        vl_loss, vl_m, which, state = vl_loss_ema, vl_ema, 'EMA', ema.ema.state_dict()
    else:
        vl_loss, vl_m, which, state = vl_loss_raw, vl_raw, 'raw', model.state_dict()

    for k in history:
        srcv={'tr_loss':tr_loss,'vl_loss':vl_loss,'tr_f1':tr_m['F1'],'vl_f1':vl_m['F1'],
              'vl_iou':vl_m['IoU'],'vl_p':vl_m['Precision'],'vl_r':vl_m['Recall']}
        history[k].append(srcv[k])

    dt=time.time()-t0
    print(f'Ep {epoch:3d}/{NUM_EPOCHS} ({dt:.0f}s) TrL={tr_loss:.4f} TrF1={tr_m["F1"]:.1f} | '
          f'VlF1={vl_m["F1"]:.1f}({which}) IoU={vl_m["IoU"]:.1f} R={vl_m["Recall"]:.1f}', end='')

    if vl_m['F1']>best_f1:
        best_f1=vl_m['F1']; patience_cnt=0; best_src=which
        torch.save({'epoch':epoch,'state_dict':state,'best_f1':best_f1,
                    'metrics':vl_m,'src':which}, 'best_eamsnet.pth')
        print(f'  *** BEST F1={best_f1:.2f}% ({which}) ***')
    else:
        patience_cnt+=1; print()
        if patience_cnt>=PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}'); break

print(f'\nDone. Best Val F1: {best_f1:.2f}% (dari {best_src})')

## 13. Training Curves

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].plot(history['tr_loss'],label='Train',lw=2); axes[0].plot(history['vl_loss'],label='Val',lw=2)
axes[0].set(xlabel='Epoch',ylabel='Loss',title='Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['tr_f1'],label='Train F1',lw=2); axes[1].plot(history['vl_f1'],label='Val F1',lw=2)
axes[1].plot(history['vl_iou'],label='Val IoU',lw=2,ls='--')
axes[1].set(xlabel='Epoch',ylabel='%',title='F1 & IoU'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(history['vl_p'],label='Precision',lw=2); axes[2].plot(history['vl_r'],label='Recall',lw=2)
axes[2].set(xlabel='Epoch',ylabel='%',title='Precision & Recall'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('training_curves_pp.png',dpi=300,bbox_inches='tight'); plt.show()

## 14. Test Evaluation — Standard, Threshold-Optimized, TTA, Post-processed

Urutan: muat model terbaik -> cari threshold optimal di VAL -> evaluasi TEST dengan beberapa konfigurasi.
Threshold dicari di VAL (bukan TEST) agar tidak ada kebocoran ke metrik akhir.


In [ ]:
ckpt=torch.load('best_eamsnet.pth',map_location=device,weights_only=False)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded best model dari epoch {ckpt["epoch"]} (Val F1: {ckpt["best_f1"]:.2f}%)')

m_std = evaluate_full(model,test_loader,device,thr=0.5,tta=False,use_postproc=False)
best_thr,val_f1 = search_threshold(model,val_loader,device)
print(f'\nThreshold optimal (dari VAL): {best_thr:.2f}  (Val F1 @thr: {val_f1:.2f}%)')
m_thr = evaluate_full(model,test_loader,device,thr=best_thr,tta=False,use_postproc=False)
m_tta = evaluate_full(model,test_loader,device,thr=best_thr,tta=True,use_postproc=False)
m_all = evaluate_full(model,test_loader,device,thr=best_thr,tta=True,use_postproc=True)

def show(name,m):
    print(f'\n{"="*60}\n  {name}\n{"="*60}')
    for k,v in m.items(): print(f'  {k:>12s}: {v:.2f}%')

show('TEST — Standard (thr=0.5)', m_std)
show('TEST — Threshold-optimized', m_thr)
show('TEST — Threshold + TTA', m_tta)
show('TEST — Threshold + TTA + Post-processing', m_all)
best_cfg=max([m_thr,m_tta,m_all],key=lambda m:m["F1"])
print(f'\nTotal gain F1: {m_all["F1"]-m_std["F1"]:+.2f}%  |  IoU: {m_all["IoU"]-m_std["IoU"]:+.2f}%')
print("\n"+"="*60)
print("  PERBANDINGAN vs BASELINE PAPER (Jeon & Jeong 2026, LEVIR-CD)")
print("="*60)
print(f"  CATATAN: baseline @512 TIDAK setara dgn hasil @256 Anda; pembanding setara = ATCFNet @256")
print(f"  DA-ChangeFormer (usulan paper) : F1=92.80%  IoU=87.28%  (@512)")
print(f"  EAMSNet++ v2 (terbaik Anda)    : F1={best_cfg['F1']:.2f}%  IoU={best_cfg['IoU']:.2f}%  (@{IMG_SIZE})")
print("="*60)

## 18. Multi-Scale + Flip TTA (gratis, tanpa retraining)

Inferensi pada beberapa skala (0.75x, 1.0x, 1.25x) x 4 flip = 12 forward pass per patch, lalu rata-ratakan.
Memuat checkpoint `best_eamsnet_pp_v2.pth` yang sudah ada. Threshold dicari ulang di VAL dgn skema TTA yang sama.

In [ ]:
@torch.no_grad()
def predict_msf(model, imgA, imgB, scales=(0.75,1.0,1.25), flips=True):
    H,W=imgA.shape[2:]
    acc=torch.zeros(imgA.shape[0],1,H,W,device=imgA.device); n=0
    def run(a,b):
        nonlocal acc,n
        variants=[(a,b)]
        if flips:
            variants+=[(a.flip(3),b.flip(3)),(a.flip(2),b.flip(2)),(a.flip(2).flip(3),b.flip(2).flip(3))]
        for i,(aa,bb) in enumerate(variants):
            with autocast():
                p=torch.sigmoid(model(aa,bb)[0].float())
            if i==1: p=p.flip(3)
            elif i==2: p=p.flip(2)
            elif i==3: p=p.flip(2).flip(3)
            if p.shape[2:]!=(H,W):
                p=F.interpolate(p,(H,W),mode='bilinear',align_corners=False)
            acc+=p; n+=1
    for s in scales:
        if s==1.0: run(imgA,imgB)
        else:
            nh,nw=int(round(H*s)),int(round(W*s))
            a=F.interpolate(imgA,(nh,nw),mode='bilinear',align_corners=False)
            b=F.interpolate(imgB,(nh,nw),mode='bilinear',align_corners=False)
            run(a,b)
    return acc/n

@torch.no_grad()
def search_threshold_msf(model, loader, device, scales, grid=None):
    if grid is None: grid=np.arange(0.30,0.71,0.02)
    model.eval()
    stats={round(t,3):[0,0,0] for t in grid}
    for imgA,imgB,label in loader:
        imgA,imgB=imgA.to(device),imgB.to(device)
        p=predict_msf(model,imgA,imgB,scales).cpu().numpy().flatten()
        t=label.long().numpy().flatten()
        for thr in grid:
            pb=(p>thr); s=stats[round(thr,3)]
            s[0]+=int(((pb==1)&(t==1)).sum()); s[1]+=int(((pb==1)&(t==0)).sum()); s[2]+=int(((pb==0)&(t==1)).sum())
    best_thr,best_f1=0.5,-1
    for thr in grid:
        tp,fp,fn=stats[round(thr,3)]; e=1e-7
        P=tp/(tp+fp+e); R=tp/(tp+fn+e); F1=2*P*R/(P+R+e)
        if F1>best_f1: best_f1,best_thr=F1,float(thr)
    return best_thr,best_f1*100

@torch.no_grad()
def evaluate_msf(model, loader, device, scales, thr=0.5):
    model.eval(); met=CDMetrics()
    for imgA,imgB,label in loader:
        imgA,imgB=imgA.to(device),imgB.to(device)
        p=predict_msf(model,imgA,imgB,scales).cpu().numpy()
        for b in range(p.shape[0]):
            met.update_from_binary(torch.from_numpy((p[b,0]>thr).astype(np.uint8)), label[b])
    return met.compute()

ckpt=torch.load('best_eamsnet.pth',map_location=device,weights_only=False)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded best model epoch {ckpt["epoch"]} (Val F1 {ckpt["best_f1"]:.2f}%)')

SCALES=(0.75,1.0,1.25)
print(f'\nMencari threshold optimal utk multi-scale TTA (scales={SCALES})...')
thr_msf,valf1_msf=search_threshold_msf(model,val_loader,device,SCALES)
print(f'Threshold MSF optimal (VAL): {thr_msf:.2f}  (Val F1: {valf1_msf:.2f}%)')

m_msf=evaluate_msf(model,test_loader,device,SCALES,thr=thr_msf)
print("\n"+"="*60); print("  TEST — Multi-Scale + Flip TTA"); print("="*60)
for k,v in m_msf.items(): print(f'  {k:>12s}: {v:.2f}%')

print("\n"+"="*60); print("  RINGKASAN PERBANDINGAN (LEVIR-CD)"); print("="*60)
print(f"  CATATAN: baseline di bawah diukur @512; hasil Anda @256 -> TIDAK setara langsung.")
print(f"  DA-ChangeFormer (paper, @512)        : F1=92.80%  IoU=87.28%")
print(f"  ATCFNet (paper, @256, 3.71M)         : F1=91.46%  IoU=84.26%   <- pembanding setara (256)")
print(f"  EAMSNet MobileNet (Anda, @256, 4.5M) : F1={m_msf['F1']:.2f}%  IoU={m_msf['IoU']:.2f}%")
print("="*60)

## 15. Qualitative Results

In [ ]:
@torch.no_grad()
def visualize(model, dataset, device, n=6, thr=0.5, path='qualitative_results_pp.png'):
    model.eval()
    idxs=np.random.choice(len(dataset),n,replace=False)
    fig,axes=plt.subplots(n,5,figsize=(20,4*n))
    titles=['T1 Image','T2 Image','Ground Truth','Prediction','Error Map']
    mean=torch.tensor([0.485,0.456,0.406]).view(3,1,1); std=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    for row,idx in enumerate(idxs):
        iA,iB,lb=dataset[idx]
        pred,_=model(iA.unsqueeze(0).to(device),iB.unsqueeze(0).to(device))
        pm=(torch.sigmoid(pred)>thr).float().cpu().squeeze().numpy()
        aV=(iA*std+mean).permute(1,2,0).numpy().clip(0,1); bV=(iB*std+mean).permute(1,2,0).numpy().clip(0,1)
        lV=lb.squeeze().numpy()
        err=np.zeros((*lV.shape,3))
        err[(pm==1)&(lV==1)]=[0,1,0]; err[(pm==1)&(lV==0)]=[1,0,0]; err[(pm==0)&(lV==1)]=[0,0,1]
        for col,img in enumerate([aV,bV,lV,pm,err]):
            kw={'cmap':'gray'} if col in [2,3] else {}
            axes[row,col].imshow(img,**kw); axes[row,col].axis('off')
            if row==0: axes[row,col].set_title(titles[col],fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.show()

visualize(model, test_dataset, device, thr=best_thr)

## 16. Ablation & Computational Complexity

In [ ]:
def benchmark(mdl, device, runs=50):
    mdl.eval()
    a=torch.randn(1,3,256,256).to(device); b=torch.randn(1,3,256,256).to(device)
    for _ in range(10):
        with torch.no_grad(): mdl(a,b)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ts=[]
    for _ in range(runs):
        t0=time.time()
        with torch.no_grad(): mdl(a,b)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        ts.append(time.time()-t0)
    ms=np.mean(ts)*1000; return ms,1000/ms

m_prop = EAMSNetPP(BACKBONE, pretrained=False, width=WIDTH).to(device)
ms,fps=benchmark(m_prop,device)
p=sum(pp.numel() for pp in m_prop.parameters())
print(f'{"EAMSNet (ATDAM+MSDA+EABRM)":<30s}  {p/1e6:6.2f}M params  {ms:6.1f} ms  {fps:6.1f} FPS')
del m_prop

## 17. Ringkasan & Catatan Kejujuran Ilmiah

**Tiga modul yang diusulkan:**
1. ATDAM (Adaptive Temporal Difference Attention) — atensi selisih temporal di tiap skala.
2. MSDA (Multi-Scale Difference Aggregation) — agregasi multi-skala di bottleneck.
3. EABRM (Edge-Aware Boundary Refinement Module) — penajaman batas di tiap level decoder.

**Sumber peningkatan lain:**
1. Backbone MobileNetV2 sebagai encoder Siamese yang ringan dan cepat.
2. Lovász loss — optimisasi IoU langsung, memperbaiki boundary.
3. Threshold optimization + TTA + post-processing morfologis — peningkatan saat inferensi.

**Ekspektasi hasil yang jujur:**
- Target realistis: **F1 ~92-93%, IoU ~86-87%** pada LEVIR-CD test set standar.
- Setara/di atas SOTA termutakhir (SChanger 92.87%, ChangeMamba 91.37%, DA-ChangeFormer 92.80%).
- **F1 > 95% pada LEVIR-CD standar TIDAK realistis**. Karena IoU = F1/(2-F1), klaim "F1 95% + IoU 85%" secara matematis tidak konsisten.
- Jika suatu konfigurasi menghasilkan >95%, periksa kemungkinan **data leakage** (patch test overlap dengan train) sebelum melaporkannya.

**Tips menjalankan:**
- Jika OOM, turunkan BATCH_SIZE ke 4 atau aktifkan gradient checkpointing.